In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from sklearn.metrics import silhouette_samples

# Note: Ensure your core math functions (factorcal, soc, imc2, slht)
# are imported here if you moved them to a separate file, e.g.:
# from clustering_core import factorcal, soc, imc2, slht

def load_image_from_url(url):
    """Fetches an image from a URL and returns an RGB numpy array."""
    response = requests.get(url)
    response.raise_for_status()

    image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    img_bgr = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return img_rgb

def main():
    image_url = input("Enter direct image URL: ")
    try:
        a = load_image_from_url(image_url)

        # 1. Proportional Resize (Fixes physical squishing)
        max_dim = 200
        height, width = a.shape[:2]
        scale = max_dim / max(height, width)
        new_width, new_height = int(width * scale), int(height * scale)
        a = cv2.resize(a, (new_width, new_height))

        # 2. Spatial Smoothing (Fixes algorithmic noise/fragmentation)
        a = cv2.GaussianBlur(a, (7, 7), 0)

    except Exception as e:
        print(f"Error loading image from URL: {e}")
        return

    f, h, k_channels = a.shape
    nk = int(input("No. of clusters required: "))
    n = f * h

    # Reshape image to (n x 3) array
    x = a.reshape((n, 3)).astype(float)

    print("-> Running SOC (Self-Optimal Clustering) with Lagrange Optimization...")
    fac = factorcal(x, nk, iter_max=10)
    result = soc(x, nk, fac)

    print("-> Running IMC-1 (Fixed Threshold Baseline)...")
    res1 = imc2(x, nk, 1.0)

    print("-> Running IMC-2 (Heuristic Threshold Baseline)...")
    res2 = imc2(x, nk, nk / (nk + 1.0))

    print("-> Calculating metrics...")

    # Calculate SOC Metrics
    s = silhouette_samples(x, result['idx'])
    S, GSS = slht(s, result['idx'], result['n'], result['m'], nk)
    print(f"GSS (SOC):   {GSS:.4f}")

    # Calculate IMC-1 Metrics
    s1 = silhouette_samples(x, res1['idx'])
    S1, GSI1 = slht(s1, res1['idx'], res1['n'], res1['m'], nk)
    print(f"GSI1 (IMC1): {GSI1:.4f}")

    # Calculate IMC-2 Metrics
    s2 = silhouette_samples(x, res2['idx'])
    S2, GSI2 = slht(s2, res2['idx'], res2['n'], res2['m'], nk)
    print(f"GSI2 (IMC2): {GSI2:.4f}")

    # --- Visual Generation ---
    metrics = [GSS, GSI1, GSI2]
    methods = ['SOC', 'IMC1', 'IMC2']
    results = [result, res1, res2]

    plt.figure(figsize=(15, 10))
    plt.style.use('seaborn-v0_8-whitegrid')

    for m_idx in range(3):
        res = results[m_idx]
        labels = res['idx']

        # Compute cluster means to recolor the image
        cmap = np.zeros((nk, 3))
        for v in range(nk):
            pts = x[labels == v]
            if len(pts) > 0:
                cmap[v, :] = np.mean(pts, axis=0)

        # Rebuild segmented image
        seg_rgb = cmap[labels]
        seg_img = seg_rgb.reshape((f, h, 3)).astype(np.uint8)

        # Plot Original
        plt.subplot(2, 3, m_idx + 1)
        plt.imshow(a)
        plt.title(f"{methods[m_idx]} - Original")
        plt.axis('off')

        # Plot Segmented
        plt.subplot(2, 3, m_idx + 4)
        plt.imshow(seg_img)
        plt.title(f"{methods[m_idx]} - Segmented")
        plt.text(10, 20, f"GSI = {metrics[m_idx]:.4f}",
                 color='yellow', fontsize=12, fontweight='bold',
                 backgroundcolor='black')
        plt.axis('off')

    plt.tight_layout()
    plt.savefig('segmentation_results.png', dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()